In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from torch.utils.data import TensorDataset, DataLoader


In [2]:
df = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/train.csv')
df2 = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/test.csv')

df['is_test'] = 0
df2['is_test'] = 1

full = pd.concat([df, df2])
full = full.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)
full.shape

(227656, 11)

In [3]:
full['is_first_txn'] = (full['hours_since_prev_txn'] == 999).astype(int)
full['hours_since_prev_txn'] = full['hours_since_prev_txn'].replace(999, np.nan)

#spending spike
full['running_total'] = full.groupby('user_id')['amount'].cumsum()
full['past_total'] = full['running_total'] - full['amount']
full['past_count'] = full.groupby('user_id').cumcount()
full['past_avg_amount'] = full['past_total'] / full['past_count']
full['amount_ratio'] = full['amount'] / full['past_avg_amount']

#new device/country/merchant
full['is_new_device'] = (full.groupby(['user_id', 'device_id']).cumcount() == 0).astype(int)
full['is_new_country'] = (full.groupby(['user_id', 'country']).cumcount() == 0).astype(int)
full['is_new_merchant'] = (full.groupby(['user_id', 'merchant_category']).cumcount() == 0).astype(int)

full['prev_gap_1'] = full.groupby('user_id')['hours_since_prev_txn'].shift(1)
full['prev_gap_2'] = full.groupby('user_id')['hours_since_prev_txn'].shift(2)
full['avg_gap_last3'] = full[['hours_since_prev_txn', 'prev_gap_1', 'prev_gap_2']].mean(axis=1)

#clean
full = full.drop(columns=['running_total', 'past_total', 'past_count', 'prev_gap_1', 'prev_gap_2'])

full.shape

(227656, 18)

In [4]:
full['hours_since_prev_txn'] = full['hours_since_prev_txn'].fillna(0)
full['amount_ratio'] = full['amount_ratio'].fillna(1)

full['amount_log'] = np.log1p(full['amount'])
full['gap_log'] = np.log1p(full['hours_since_prev_txn'])
full['ratio_log'] = np.log1p(full['amount_ratio'])

full['amount_log'] = (full['amount_log'] - full['amount_log'].mean()) / full['amount_log'].std()
full['gap_log'] = (full['gap_log'] - full['gap_log'].mean()) / full['gap_log'].std()
full['ratio_log'] = (full['ratio_log'] - full['ratio_log'].mean()) / full['ratio_log'].std()

seq_cols = ['amount_log', 'gap_log', 'ratio_log', 'channel', 'is_first_txn', 'is_new_device', 'is_new_country', 'is_new_merchant']


In [5]:
full[seq_cols].describe()

,amount_log,gap_log,ratio_log,channel,is_first_txn,is_new_device,is_new_country,is_new_merchant
count,2.276560e+05,2.276560e+05,2.276560e+05,227656.000000,227656.000000,227656.000000,227656.000000,227656.000000
mean,3.400777e-16,1.273419e-16,-3.028739e-16,0.299201,0.020127,0.040684,0.099145,0.250633
std,1.000000e+00,1.000000e+00,1.000000e+00,0.457909,0.140435,0.197558,0.298858,0.433378
min,-3.767361e+00,-5.260750e+00,-2.460638e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-6.937878e-01,-2.747546e-01,-7.236710e-01,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-2.607568e-02,2.002777e-01,-1.058544e-01,0.000000,0.000000,0.000000,0.000000,0.000000
75%,6.671579e-01,5.687051e-01,5.761041e-01,1.000000,0.000000,0.000000,0.000000,1.000000
max,4.642287e+00,1.825309e+00,7.783841e+00,1.000000,1.000000,1.000000,1.000000,1.000000


In [6]:
full['present'] = 1
seq_cols = ['amount_log', 'gap_log', 'ratio_log', 'channel', 'is_first_txn', 'is_new_device', 'is_new_country', 'is_new_merchant', 'present']

steps = []
for k in range(19, -1, -1):
    shifted = full.groupby('user_id')[seq_cols].shift(k).fillna(0).to_numpy()
    steps.append(shifted)
    
x_seq = np.stack(steps, axis=1)

In [7]:
x_seq.shape

(227656, 20, 9)

In [8]:
x_seq[0]

array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0

In [9]:
labeled_idx = np.where(full['is_test'] == 0)[0]
y_labeled = full[full['is_test'] == 0]['label'].to_numpy()

train_idx, val_idx = train_test_split(labeled_idx, test_size=0.2, stratify=y_labeled, random_state=42)

x_train_seq = x_seq[train_idx]
x_val_seq = x_seq[val_idx]

all_labels = full['label'].to_numpy()
y_train_seq = all_labels[train_idx]
y_val_seq = all_labels[val_idx]


In [10]:
x_train_seq.shape

(145700, 20, 9)

In [11]:
x_val_seq.shape

(36425, 20, 9)

In [12]:
y_train_seq.mean()

np.float64(0.016417295813315032)

In [13]:
y_val_seq.mean()

np.float64(0.016417295813315032)

In [17]:
xtr = torch.tensor(x_train_seq, dtype=torch.float32)
xva = torch.tensor(x_val_seq, dtype=torch.float32)
ytr = torch.tensor(y_train_seq, dtype=torch.float32)
yva = torch.tensor(y_val_seq, dtype=torch.float32)

train_ds = TensorDataset(xtr, ytr)
val_ds = TensorDataset(xva, yva)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)

In [18]:
xb, yb = next(iter(train_loader))
xb.shape

torch.Size([512, 20, 9])

In [19]:
yb.shape

torch.Size([512])